[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/thermal_conduction.ipynb)

# Steady-State Thermal Conduction Homogenization

A walkthrough of FFTjax's thermal conduction solver (`problems.thermal.solve_thermal`) -- the
scalar (heat conduction) analogue of [`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb)'s
elastic solve, mirrored structurally throughout: build $K(\mathbf{x})$ instead of
$\mathbb{C}(\mathbf{x})$, prescribe a macroscopic temperature gradient instead of a macroscopic
strain, solve for a periodic fluctuation instead of a periodic displacement.

We want to solve the thermal equilibrium problem on a periodic voxel grid subject to its governing
PDE constraints:

$$
\nabla \cdot \mathbf{q}(\mathbf{x}) = 0, \qquad
\mathbf{q} = -K(\mathbf{x})\,\nabla T, \qquad
\nabla T = \overline{\nabla T} + \nabla T'(\mathbf{x})
$$

where $\overline{\nabla T}$ is the prescribed macroscopic temperature gradient and $T'$ is an
unknown periodic fluctuation (mean zero -- an arbitrary gauge choice, only $\nabla T'$ is
physically meaningful). One difference from the elastic case worth flagging up front: a temperature
gradient/flux pair is a plain 3-vector, not a symmetric 2nd-order tensor like strain/stress --
there's no Voigt-6 form for it (`post.fields.macroscopic_thermal_response` reports plain
`(grad_T_bar, flux_bar)` vectors, not a 6-component `to_voigt` pair).

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from materialmodels.thermal.isotropic import ThermalConductivityIsotropic
from problems.thermal import solve_thermal
from post.fields import macroscopic_thermal_response, vector_field_to_grid

## A two-phase laminate — closed-form validation

Before trusting the solver on a real microstructure, check it against a case with a known answer.
A laminate (layers of two materials, normal to $x$) has a textbook effective conductivity:

- gradient **normal** to the layers ($x$, a *series* thermal circuit) — the **harmonic mean**
  $k_\text{eff} = \big(\tfrac{v_1}{k_1} + \tfrac{v_2}{k_2}\big)^{-1}$
- gradient **in-plane** ($y$, a *parallel* circuit) — the **arithmetic mean**
  $k_\text{eff} = v_1 k_1 + v_2 k_2$

Same validation `test/test_solvers_elliptic_scalar_thermal.py` runs as a permanent regression test;
here it's worked through interactively.

In [ ]:
n = (64, 8, 8)
L = (1.0, 1.0, 1.0)   # any consistent length unit
Nv = int(np.prod(n))

# Layers normal to x, equal volume fraction.
phase_np = np.zeros(n, dtype=int)
phase_np[n[0] // 2:, :, :] = 1
phase = jnp.array(phase_np.reshape(-1))

# W/(m*K) -- representative values for epoxy resin and E-glass fibre (used again for the
# composite RVE below, so both sections of this notebook share one physically grounded pair
# instead of arbitrary round numbers).
k_matrix, k_fiber = 0.20, 1.05
materials = [ThermalConductivityIsotropic(k_matrix, "phase 0"), ThermalConductivityIsotropic(k_fiber, "phase 1")]

vf = 0.5
k_eff_series   = 1.0 / (vf / k_matrix + vf / k_fiber)
k_eff_parallel = vf * k_matrix + vf * k_fiber
print(f"analytical k_eff:  series (harmonic mean) = {k_eff_series:.4f}   "
      f"parallel (arithmetic mean) = {k_eff_parallel:.4f}")


In [ ]:
# gradient along x -- normal to the layers, a series circuit
grad_T_bar_x = jnp.array([1.0, 0.0, 0.0])
results = solve_thermal(n, L, phase, materials, grad_T_bar_x, toler_lin=1e-10, maxiter=2000)
sol = results[0].solution
flux_bar = jnp.mean(sol.flux, axis=-1)
k_eff_x = -float(flux_bar[0]) / float(grad_T_bar_x[0])
print(f"converged: {bool(sol.converged)}")
print(f"k_eff (numeric, series)   = {k_eff_x:.6f}   (analytical {k_eff_series:.6f})")
assert abs(k_eff_x - k_eff_series) < 1e-4

# gradient along y -- in-plane, a parallel circuit
grad_T_bar_y = jnp.array([0.0, 1.0, 0.0])
results = solve_thermal(n, L, phase, materials, grad_T_bar_y, toler_lin=1e-10, maxiter=2000)
sol = results[0].solution
flux_bar = jnp.mean(sol.flux, axis=-1)
k_eff_y = -float(flux_bar[1]) / float(grad_T_bar_y[1])
print(f"k_eff (numeric, parallel) = {k_eff_y:.6f}   (analytical {k_eff_parallel:.6f})")
assert abs(k_eff_y - k_eff_parallel) < 1e-4

print("\nBoth bounds match to within CG tolerance.")

## Mixed gradient/flux boundary conditions

`control` marks which directions are flux-controlled (1) rather than gradient-controlled (0) --
the scalar analogue of `solve_mechanics`'s mixed strain/stress `control`. Driving the gradient
along $x$ while prescribing a nonzero target flux on $y$ and $z$ (instead of leaving them at
whatever gradient the series circuit implies) is solved for jointly, and the solved macroscopic
gradient on those directions comes back in `grad_T_bar_out`.

In [ ]:
control = (0, 1, 1)                        # y, z flux-controlled; x gradient-controlled
flux_goal = jnp.array([0.0, -0.7, 0.3])    # target flux on y, z

results = solve_thermal(n, L, phase, materials, grad_T_bar_x,
                         control=control, flux_goal=flux_goal, toler_lin=1e-10, maxiter=3000)
sol = results[0].solution
flux_bar = jnp.mean(sol.flux, axis=-1)

print("flux_bar        =", np.asarray(flux_bar))
print("target flux_goal=", np.asarray(flux_goal), " (y, z should match)")
print("grad_T_bar_out   =", np.asarray(sol.grad_T_bar), " (x stays 1.0, y/z solved for)")

assert abs(float(flux_bar[1]) - float(flux_goal[1])) < 1e-4
assert abs(float(flux_bar[2]) - float(flux_goal[2])) < 1e-4
assert abs(float(sol.grad_T_bar[0]) - 1.0) < 1e-10
print("\nFlux-controlled directions hit their targets; the gradient-controlled entry is untouched.")

## A real composite microstructure

Same square-packed fibre RVE as `lin-elastic_strain.ipynb` -- a matrix phase with circular fibre
cross-sections on a square lattice. No closed-form bound applies to a real (non-laminate)
microstructure, but the effective conductivity along any direction must still fall between the
series (harmonic-mean) and parallel (arithmetic-mean) bounds above -- a real material can never do
better than an idealized parallel circuit or worse than an idealized series one.

Two geometric caveats worth being explicit about, since the number below turns out **not** to land
in the published range quoted below:

- This RVE is a 2-D cross-section extruded one voxel along $z$ (`nz=1`), with fibres running along
  $z$. A gradient prescribed in the $x$-$y$ plane is **transverse** conduction within a single ply
  -- heat crossing from fibre to resin to fibre -- not conduction *along* the fibre length.
- For a real unidirectional glass-fibre/epoxy laminate at ~50% Vf, published values are roughly:

  | direction | conductivity |
  |---|---|
  | in-plane, along the fibres | 0.55 - 0.65 W/(m*K) |
  | through-plane (across the laminate thickness, i.e. across many stacked plies) | 0.30 - 0.36 W/(m*K) |

  "Through-plane" there means across the *laminate thickness* -- through several plies and their
  resin-rich interlaminar layers, plus whatever void content the manufacturing process leaves
  behind (air is a strong insulator, ~0.026 W/(m*K), so a few percent void content measurably
  lowers the bulk number). This RVE models neither of those -- it's the *idealized, single-ply,
  void-free* transverse conductivity between circular fibres on a perfect square lattice, so it's
  reasonable for it to land somewhat *above* the real laminate's through-plane number, not
  reproduce it exactly. Bring in a multi-ply RVE (or a phase for void content) to close that gap.

In [ ]:
from generation.rve import make_square_composite_rve

phi, r_fiber, dx = 0.5, 0.005, 0.0002
phase_np, n, L, phi_act = make_square_composite_rve(
    phi=phi, r_fiber=r_fiber, dx=dx, N_min=32, nz=1,
)
Nv = int(np.prod(n))
phase = jnp.array(phase_np.reshape(-1))
print("grid n :", n, "  fibre volume fraction (actual):", f"{phi_act:.4f}")

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r",
          extent=(0, n[0] * dx * 1000, 0, n[1] * dx * 1000))
ax.set_title(f"Fibre cross-section (Vf={phi_act:.3f})")
ax.set_xlabel("x [µm]")
ax.set_ylabel("y [µm]")
ax.set_aspect("equal")
plt.show()

In [ ]:
materials = [ThermalConductivityIsotropic(k_matrix, "matrix"), ThermalConductivityIsotropic(k_fiber, "fibre")]

grad_T_bar = jnp.array([1.0, 0.0, 0.0])
results = solve_thermal(n, L, phase, materials, grad_T_bar, toler_lin=1e-8)
sol = results[0].solution
print("converged:", bool(sol.converged))

grad_bar = jnp.mean(sol.grad_T, axis=-1)
flux_bar = jnp.mean(sol.flux, axis=-1)
resp = macroscopic_thermal_response(grad_bar, flux_bar)
print("macroscopic response:", {k: (round(float(v), 5) if np.ndim(v) == 0 else np.round(v, 5))
                                  for k, v in resp.items()})

k_eff_x = -float(flux_bar[0]) / float(grad_T_bar[0])
print(f"\nk_eff_x (single-ply transverse, this RVE) = {k_eff_x:.4f} W/(m*K)")
print(f"  series/parallel bounds                  : [{k_eff_series:.4f}, {k_eff_parallel:.4f}]  (satisfied)")
print(f"  published laminate through-plane range  : [0.30, 0.36]  -- {k_eff_x:.4f} is ABOVE it, see markdown above")
assert k_eff_series <= k_eff_x <= k_eff_parallel


In [ ]:
# Local flux magnitude |q(x)| -- concentrates in the stiffer, higher-conductivity fibre and its
# immediate neighbourhood, the thermal analogue of the stress concentration lin-elastic_strain.ipynb
# shows for von Mises stress.
flux_grid = vector_field_to_grid(sol.flux, n)
flux_mag = np.linalg.norm(flux_grid, axis=-1)[:, :, 0]

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(flux_mag.T, origin="lower", cmap="inferno",
               extent=(0, n[0] * dx * 1000, 0, n[1] * dx * 1000))
ax.set_title("Local flux magnitude $|\\mathbf{q}(\\mathbf{x})|$")
ax.set_xlabel("x [µm]")
ax.set_ylabel("y [µm]")
ax.set_aspect("equal")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## Anisotropic fibre conductivity: along vs. across the fibre axis

Real fibres are often not isotropic -- carbon fibre in particular can have a conductivity along its
own axis many times its transverse value (E-glass, used above, happens to be close to isotropic,
which is why the plain `ThermalConductivityIsotropic` fibre already gave the right transverse
answer). `materialmodels.thermal.transverse_isotropic.ThermalConductivityTransverseIsotropic` --
mirroring the elastic side's `TransverseIsotropic` -- carries the two independent constants a
fibre's conductivity actually needs: `k_L` along its axis, `k_T` across it.

This RVE's fibres run along $z$ (`nz=1`), so both regimes can be probed on the *same* geometry, just
by changing which direction the gradient points:

- gradient in the $x$-$y$ plane -- **transverse** conduction, governed by `k_T` only (bit-identical
  to `ThermalConductivityIsotropic(k_T)` regardless of `k_L`, verified in
  `test/test_materialmodels_thermal_transverse_isotropic.py`)
- gradient along $z$ -- **along-fibre** conduction, governed by `k_L`

The fibre below (`k_L=20`, `k_T=1.5`, arbitrary units) is deliberately more anisotropic than
E-glass, to make the two regimes visibly different -- illustrative, not a claim about any specific
real fibre's measured properties.

In [ ]:
from materialmodels.thermal.transverse_isotropic import ThermalConductivityTransverseIsotropic

k_L_fiber, k_T_fiber = 20.0, 1.5   # illustrative (carbon-fibre-like anisotropy), not measured data

materials_aniso = [
    ThermalConductivityIsotropic(k_matrix, "matrix"),
    ThermalConductivityTransverseIsotropic(k_L=k_L_fiber, k_T=k_T_fiber,
                                            fiber_dir=[0.0, 0.0, 1.0], name="fibre (anisotropic)"),
]

# transverse: gradient in-plane, same direction as the composite section above
results_T = solve_thermal(n, L, phase, materials_aniso, grad_T_bar, toler_lin=1e-10, maxiter=3000)
flux_T = jnp.mean(results_T[0].solution.flux, axis=-1)
k_eff_T = -float(flux_T[0]) / float(grad_T_bar[0])

# along-fibre: gradient along z
grad_T_bar_z = jnp.array([0.0, 0.0, 1.0])
results_L = solve_thermal(n, L, phase, materials_aniso, grad_T_bar_z, toler_lin=1e-10, maxiter=3000)
sol_L = results_L[0].solution
flux_L = jnp.mean(sol_L.flux, axis=-1)
k_eff_L = -float(flux_L[2]) / float(grad_T_bar_z[2])

print(f"transverse  k_eff (x-y plane, uses k_T={k_T_fiber})  : {k_eff_T:.4f}")
print(f"along-fibre k_eff (z, uses k_L={k_L_fiber})          : {k_eff_L:.4f}")
print(f"anisotropy ratio k_eff_L / k_eff_T                   : {k_eff_L / k_eff_T:.2f}")

Along the fibre axis the extruded microstructure doesn't vary with $z$, so the trivial
ansatz $T'=0$ (no periodic fluctuation at all) already satisfies $\nabla\cdot\mathbf{q}=0$ exactly:
the flux $q_z(x,y) = -K_{zz}(x,y)\,\overline{\nabla T}_z$ varies with $(x,y)$, but its divergence
along $z$ is zero regardless, since neither $K$ nor the prescribed gradient depend on $z$. That
makes the along-fibre effective conductivity an *exact*, closed-form volume-fraction-weighted
arithmetic mean -- a stronger check than the CG-tolerance-limited laminate bounds earlier.

In [ ]:
k_eff_z_analytical = phi_act * k_L_fiber + (1 - phi_act) * k_matrix
print(f"k_eff_L (numeric)    = {k_eff_L:.6f}")
print(f"k_eff_L (analytical) = {k_eff_z_analytical:.6f}   (Vf*k_L + (1-Vf)*k_matrix)")
print(f"max|T_prime| along z = {float(jnp.max(jnp.abs(sol_L.T_prime))):.2e}   (expect ~0 -- trivial solution)")

assert abs(k_eff_L - k_eff_z_analytical) < 1e-6
assert float(jnp.max(jnp.abs(sol_L.T_prime))) < 1e-8
print("\nPASSED: along-fibre conduction matches the exact closed form.")

## Next steps

- Swap in a mix of fibre orientations or a third phase -- `materialmodels.assembly.assemble_K_field`
  is generic over any `ConductivityModel`, no different from the elastic side's `assemble_C_field`.
- Sweep the gradient direction over `uniaxial_x/y/z` (see `src/utils/loadcases.py`'s registry, built
  for exactly this on the elastic side) to assemble a full effective conductivity tensor from several
  single-direction solves.
- Give the anisotropic fibre a per-voxel orientation field instead of one global `fiber_dir` --
  `ThermalConductivityTransverseIsotropic.conductivity_field_oriented` (and `assemble_K_field`'s
  automatic detection of it) already support that, only this notebook's RVE happens to be
  single-orientation.
- `solve_thermal`'s `writer=` argument writes `gradient`/`flux`/`temperature_fluctuation` fields to
  XDMF/HDF5 via the same `utils.io.xdmf_writer.IncrementalWriter` every other solve in this project
  uses -- open the result in ParaView exactly like an elastic or fracture run.